In [ ]:
import pandas as pd
import numpy as np
from engine import apply_filters
from presets import load_presets, run_all_presets, validate_preset_counts


In [ ]:
df= pd.read_csv(r"C:\Users\bhrra\Desktop\nifty100\data\processed\financial_ratios.csv")

In [ ]:
df_copy = df.copy()

In [ ]:
financial_ratios_df = df_copy

In [ ]:
print(financial_ratios_df.columns.tolist())

In [ ]:
import sqlite3
# import pandas as pd
from pathlib import Path
path = Path(r"C:\Users\bhrra\Desktop\nifty100\db\nifty100.db")
conn = sqlite3.connect(path)

# Sabhi tables ki list
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

for table_name in tables["name"]:
    cols = pd.read_sql(f"PRAGMA table_info({table_name});", conn)
    print(f"\n--- {table_name} ---")
    print(cols["name"].tolist())

In [ ]:


conn = sqlite3.connect(path)
YEAR = 2025  # jo bhi latest year screen karna hai

# --- financial_ratios: core ratios, already latest-year filtered ---
fr = pd.read_sql(f"SELECT * FROM financial_ratios WHERE year = {YEAR}", conn)

# --- sectors: broad_sector, company_id ke against (year nahi hai isme) ---
sectors = pd.read_sql("SELECT company_id, broad_sector FROM sectors", conn)

# --- profitandloss: Sales chahiye Debt-Free Blue Chip preset ke liye ---
pnl = pd.read_sql(f"SELECT company_id, sales, net_profit FROM profitandloss WHERE year = {YEAR}", conn)

# --- market_cap: PE, PB, Dividend Yield, Market Cap — Value Pick preset ke liye ---
mcap = pd.read_sql(f"SELECT company_id, pe_ratio, pb_ratio, dividend_yield_pct, market_cap_crore FROM market_cap WHERE year = {YEAR}", conn)

# --- Sab merge karo company_id pe ---
screener_input_df = (
    fr
    .merge(sectors, on="company_id", how="left")
    .merge(pnl, on="company_id", how="left")
    .merge(mcap, on="company_id", how="left")
)

print(screener_input_df.shape)
print(screener_input_df.columns.tolist())

In [ ]:
RENAME_MAP = {
    "return_on_equity_pct": "ROE",
    "debt_to_equity": "DE",
    "interest_coverage": "ICR",          # numeric value
    "icr_label": "ICR_label",            # "Debt Free" text yahan hai
    "free_cash_flow_cr": "FCF",
    "operating_profit_margin_pct": "OPM",
    "asset_turnover": "Asset_Turnover",
    "dividend_payout_ratio_pct": "Dividend_Payout",
    "net_profit_margin_pct": "NPM",
    "earnings_per_share": "EPS",
    "cash_from_operations_cr": "CFO",
    "revenue_cagr_5yr": "Revenue_CAGR_5yr",
    "pat_cagr_5yr": "PAT_CAGR_5yr",
    "eps_cagr_5yr": "EPS_CAGR",
    "sales": "Sales",
    "net_profit": "Net_Profit",
    "pe_ratio": "PE",
    "pb_ratio": "PB",
    "dividend_yield_pct": "Dividend_Yield",
    "market_cap_crore": "Market_Cap",
}

screener_input_df = screener_input_df.rename(columns=RENAME_MAP)

In [29]:
# # run_screener.py
import sqlite3
# import pandas as pd
# import numpy as np
# from src.screener.presets import run_all_presets, validate_preset_counts

# ---------- Step 1: DB Connect ----------
# DB_PATH = r"db/nifty100.db"   # <-- apna actual .db file path yahan daalo
conn = sqlite3.connect(path)
YEAR = 2025

# ---------- Step 2: Rename Map (schema ke real column names -> engine ke expected names) ----------
RENAME_MAP = {
    "return_on_equity_pct": "ROE",
    "debt_to_equity": "DE",
    "interest_coverage": "ICR",
    "icr_label": "ICR_label",
    "free_cash_flow_cr": "FCF",
    "operating_profit_margin_pct": "OPM",
    "asset_turnover": "Asset_Turnover",
    "dividend_payout_ratio_pct": "Dividend_Payout",
    "net_profit_margin_pct": "NPM",
    "earnings_per_share": "EPS",
    "cash_from_operations_cr": "CFO",
    "revenue_cagr_5yr": "Revenue_CAGR_5yr",
    "pat_cagr_5yr": "PAT_CAGR_5yr",
    "eps_cagr_5yr": "EPS_CAGR",
    "sales": "Sales",
    "net_profit": "Net_Profit",
    "pe_ratio": "PE",
    "pb_ratio": "PB",
    "dividend_yield_pct": "Dividend_Yield",
    "market_cap_crore": "Market_Cap",
}

# ---------- Step 3: Base Tables Load & Merge ----------
fr = pd.read_sql(f"SELECT * FROM financial_ratios WHERE year = {YEAR}", conn)
sectors = pd.read_sql("SELECT company_id, broad_sector FROM sectors", conn)
pnl = pd.read_sql(f"SELECT company_id, sales, net_profit FROM profitandloss WHERE year = {YEAR}", conn)
mcap = pd.read_sql(f"SELECT company_id, pe_ratio, pb_ratio, dividend_yield_pct, market_cap_crore FROM market_cap WHERE year = {YEAR}", conn)

screener_input_df = (
    fr.merge(sectors, on="company_id", how="left")
      .merge(pnl, on="company_id", how="left")
      .merge(mcap, on="company_id", how="left")
      .rename(columns=RENAME_MAP)
)

# ---------- Step 4: Revenue CAGR 3yr (Turnaround Watch ke liye) ----------
def compute_revenue_cagr_3yr(conn: sqlite3.Connection, end_year: int) -> pd.DataFrame:
    """profitandloss se current aur 3-saal-pehle ka sales leke CAGR compute karta hai."""
    start_year = end_year - 3

    sales_end = pd.read_sql(
        f"SELECT company_id, sales AS sales_end FROM profitandloss WHERE year = {end_year}", conn
    )
    sales_start = pd.read_sql(
        f"SELECT company_id, sales AS sales_start FROM profitandloss WHERE year = {start_year}", conn
    )

    merged = sales_end.merge(sales_start, on="company_id", how="inner")

    valid = (merged["sales_start"] > 0) & (merged["sales_end"] > 0)
    merged["Revenue_CAGR_3yr"] = np.nan
    merged.loc[valid, "Revenue_CAGR_3yr"] = (
        (merged.loc[valid, "sales_end"] / merged.loc[valid, "sales_start"]) ** (1 / 3) - 1
    ) * 100

    return merged[["company_id", "Revenue_CAGR_3yr"]]


revenue_cagr_3yr_df = compute_revenue_cagr_3yr(conn, end_year=YEAR)

# ---------- Step 5: DE Current vs Previous Year (Turnaround Watch ke liye) ----------
fr_curr = pd.read_sql(
    f"SELECT company_id, debt_to_equity AS DE_current_year FROM financial_ratios WHERE year = {YEAR}", conn
)
fr_prev = pd.read_sql(
    f"SELECT company_id, debt_to_equity AS DE_previous_year FROM financial_ratios WHERE year = {YEAR - 1}", conn
)

# ---------- Step 6: SAB MERGES YAHAN COMPLETE KARO — run_all_presets() se PEHLE ----------
screener_input_df = (
    screener_input_df
    .merge(revenue_cagr_3yr_df, on="company_id", how="left")
    .merge(fr_curr, on="company_id", how="left")
    .merge(fr_prev, on="company_id", how="left")
)

print("Final columns:", screener_input_df.columns.tolist())
print("Shape:", screener_input_df.shape)

# ---------- Step 7: Ab Screener Chalao ----------
CONFIG_PATH = r"C:\Users\bhrra\Desktop\nifty100\src\Analytics\screener_config.yaml"  # <-- apna actual yaml path

results = run_all_presets(screener_input_df, CONFIG_PATH)

for name, res_df in results.items():
    print(f"{name}: {len(res_df)} companies")

print(validate_preset_counts(results))

Final columns: ['id', 'company_id', 'year', 'NPM', 'OPM', 'ROE', 'DE', 'ICR', 'Asset_Turnover', 'FCF', 'capex_cr', 'EPS', 'book_value_per_share', 'Dividend_Payout', 'total_debt_cr', 'CFO', 'ICR_label', 'capex_intensity_pct', 'capital_allocation_label', 'Revenue_CAGR_5yr', 'PAT_CAGR_5yr', 'EPS_CAGR', 'broad_sector', 'Sales', 'Net_Profit', 'PE', 'PB', 'Dividend_Yield', 'Market_Cap', 'Revenue_CAGR_3yr', 'DE_current_year', 'DE_previous_year']
Shape: (0, 32)
Quality Compounder: 0 companies
Value Pick: 0 companies
Growth Accelerator: 0 companies
Dividend Champion: 0 companies
Debt-Free Blue Chip: 0 companies
Turnaround Watch: 0 companies
                preset  count  in_range
0   Quality Compounder      0     False
1           Value Pick      0     False
2   Growth Accelerator      0     False
3    Dividend Champion      0     False
4  Debt-Free Blue Chip      0     False
5     Turnaround Watch      0     False


In [32]:


def parse_period(year_series: pd.Series) -> pd.DataFrame:
    """
    'Mar 2024', 'Dec 2023' jaise strings ko date + numeric year mein todta hai.
    Return: period_date (sortable), fiscal_year (int), period_month (3=Mar,12=Dec,6=Jun,9=Sep)
    """
    period_date = pd.to_datetime(year_series, format="%b %Y", errors="coerce")
    return pd.DataFrame({
        "period_date": period_date,
        "fiscal_year": period_date.dt.year,
        "period_month": period_date.dt.month,
    })


def get_latest_annual_snapshot(df: pd.DataFrame, year_col="year") -> pd.DataFrame:
    """
    Har company_id ke liye uska SABSE RECENT annual period (Mar ya Dec end)
    nikalta hai — Jun/Sep TTM stubs ko exclude karke.
    """
    df = df.copy()
    parsed = parse_period(df[year_col])
    df = pd.concat([df, parsed], axis=1)

    # Sirf annual fiscal-year-end periods rakho (Mar=3, Dec=12); Jun/Sep TTM chhod do
    annual_only = df[df["period_month"].isin([3, 12])]

    # Har company ka sabse latest annual period lo
    latest = (
        annual_only
        .sort_values("period_date")
        .groupby("company_id", as_index=False)
        .tail(1)
    )
    return latest

In [ ]:
# run_screener.py
import sqlite3
import pandas as pd
# from src.screener.presets import run_all_presets, validate_preset_counts

conn = sqlite3.connect(path)
YEAR = 2025

fr = pd.read_sql(f"SELECT * FROM financial_ratios WHERE year = {YEAR}", conn)
sectors = pd.read_sql("SELECT company_id, broad_sector FROM sectors", conn)
pnl = pd.read_sql(f"SELECT company_id, sales, net_profit FROM profitandloss WHERE year = {YEAR}", conn)
mcap = pd.read_sql(f"SELECT company_id, pe_ratio, pb_ratio, dividend_yield_pct, market_cap_crore FROM market_cap WHERE year = {YEAR}", conn)

screener_input_df = (
    fr.merge(sectors, on="company_id", how="left")
      .merge(pnl, on="company_id", how="left")
      .merge(mcap, on="company_id", how="left")
      .rename(columns=RENAME_MAP)   # RENAME_MAP upar wala use karo
)

# ==========
def compute_revenue_cagr_3yr(conn: sqlite3.Connection, end_year: int) -> pd.DataFrame:
    """
    profitandloss table se current year aur (end_year - 3) ka sales
    nikalke 3-year CAGR compute karta hai.
    """
    start_year = end_year - 3

    sales_end = pd.read_sql(
        f"SELECT company_id, sales AS sales_end FROM profitandloss WHERE year = {end_year}",
        conn
    )
    sales_start = pd.read_sql(
        f"SELECT company_id, sales AS sales_start FROM profitandloss WHERE year = {start_year}",
        conn
    )

    merged = sales_end.merge(sales_start, on="company_id", how="inner")

    # Zero/negative/missing start sales ko safely handle karo (CAGR undefined hoga)
    valid = (merged["sales_start"] > 0) & (merged["sales_end"] > 0)
    merged["Revenue_CAGR_3yr"] = np.nan
    merged.loc[valid, "Revenue_CAGR_3yr"] = (
        (merged.loc[valid, "sales_end"] / merged.loc[valid, "sales_start"]) ** (1 / 3) - 1
    ) * 100

    return merged[["company_id", "Revenue_CAGR_3yr"]]
revenue_cagr_3yr_df = compute_revenue_cagr_3yr(conn, end_year=YEAR)

screener_input_df = screener_input_df.merge(revenue_cagr_3yr_df, on="company_id", how="left")

# ==============
results = run_all_presets(screener_input_df, r"C:\Users\bhrra\Desktop\nifty100\src\Analytics\screener_config.yaml")

for name, res_df in results.items():
    print(f"{name}: {len(res_df)} companies")

print(validate_preset_counts(results))

fr_curr = pd.read_sql(f"SELECT company_id, debt_to_equity AS DE_current_year FROM financial_ratios WHERE year = {YEAR}", conn)
fr_prev = pd.read_sql(f"SELECT company_id, debt_to_equity AS DE_previous_year FROM financial_ratios WHERE year = {YEAR - 1}", conn)
screener_input_df = screener_input_df.merge(fr_curr, on="company_id", how="left").merge(fr_prev, on="company_id", how="left")

In [30]:
fr = pd.read_sql(f"SELECT * FROM financial_ratios WHERE year = {YEAR}", conn)
print("fr rows:", len(fr))          # <-- yeh pehle check karo

sectors = pd.read_sql("SELECT company_id, broad_sector FROM sectors", conn)
print("sectors rows:", len(sectors))

pnl = pd.read_sql(f"SELECT company_id, sales, net_profit FROM profitandloss WHERE year = {YEAR}", conn)
print("pnl rows:", len(pnl))

mcap = pd.read_sql(f"SELECT company_id, pe_ratio, pb_ratio, dividend_yield_pct, market_cap_crore FROM market_cap WHERE year = {YEAR}", conn)
print("mcap rows:", len(mcap))

fr rows: 0
sectors rows: 276
pnl rows: 0
mcap rows: 0


In [31]:
available_years = pd.read_sql("SELECT DISTINCT year FROM financial_ratios ORDER BY year", conn)
print(available_years)

        year
0   Dec 2012
1   Dec 2013
2   Dec 2014
3   Dec 2015
4   Dec 2016
5   Dec 2017
6   Dec 2018
7   Dec 2019
8   Dec 2020
9   Dec 2021
10  Dec 2022
11  Dec 2023
12  Jun 2013
13  Jun 2014
14  Jun 2015
15  Mar 2013
16  Mar 2014
17  Mar 2015
18  Mar 2016
19  Mar 2017
20  Mar 2018
21  Mar 2019
22  Mar 2020
23  Mar 2021
24  Mar 2022
25  Mar 2023
26  Mar 2024
27  Sep 2014
28  Sep 2015
29  Sep 2016
30  Sep 2017
31  Sep 2018
32  Sep 2019
33  Sep 2020
34  Sep 2021
35  Sep 2022
36  Sep 2023
37  Sep 2024


In [ ]:

# STEP A: Apna financial_ratios data load karo
# (Sprint 1-2 mein jo SQLite table banayi thi, wahan se aayega)

# ya: financial_ratios_df = pd.read_sql("SELECT * FROM financial_ratios", conn)

# STEP B: Sab presets chalao

results = run_all_presets(financial_ratios_df, r"C:\Users\bhrra\Desktop\nifty100\src\Analytics\screener_config.yaml")

# STEP C: Har preset ka result count dekho
for name, df in results.items():
    print(f"{name}: {len(df)} companies")

# STEP D: Counts valid hain ya nahi (5-50 range) check karo
summary = validate_preset_counts(results)
print(summary)

# STEP E: Ek preset ka top-5 dekho (sanity check)
print(results["Quality Compounder"].head())


# dummy_df = pd.DataFrame({
#     "company_id": [1, 2, 3, 4],
#     "broad_sector": ["IT", "Financials", "FMCG", "IT"],
#     "ROE": [18, 22, 10, 16],
#     "DE": [0.5, 8.0, 0.3, 0.9],   # note: company 2 ka DE bahut high hai, but Financials hai
#     "ICR": [12, "Debt Free", 5, 20],
#     "FCF": [100, 200, -50, 300],
#     "Revenue_CAGR_5yr": [12, 8, 15, 11],
#     "Sales": [6000, 8000, 3000, 5500],
# })

filtered = apply_filters(df_copy, {"ROE_min": 15, "DE_max": 1.0})
print(filtered)